# This script will use Evolutionary Algorithm to produce most slow fixating graphs 

In [ ]:
# imports
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process

import sys

sys.path.insert(0, "src")

import os
import numpy as np
import joblib
import pandas as pd
from moran_process.core.population_graph import PopulationGraph
# from moran_process.analysis.analysis_utils import GRAPH_PROPERTY_COLUMNS
from moran_process.analysis.analysis_utils.theory import (
    analytic_moran_fc_fixation_prob,
    analytic_moran_fc_fixation_time,
)
from pathlib import Path
from tqdm import tqdm

In [ ]:
# The batch on which we take the ML models from. It shouldn't matter much.
BATCH_NAME = "2026_07_15-respiratory-vs-random-100K-reps-2"
OVERWRITE_EXTREME_GRAPHS = True

ROOT = Path(os.getcwd())

# Now define your paths relative to ROOT
data_dir = ROOT / "simulation_data"
BATCH_DIR = data_dir / BATCH_NAME


ML_MODELS_DIR = BATCH_DIR / "ml_models_residual"
EXTREME_GRAPH_ZOO_DIR = BATCH_DIR / "extreme_graph_zoo"


In [ ]:
# Create Initial Random Graph Population
SEED = 42
rng = np.random.default_rng(SEED)

N_INITIAL_GRAPH_POPULATION = 10
NUMBER_OF_CHILDREN = 10
GENERATIONS = 100
N_NODES = 31
N_EDGES = 34


random_graph_zoo: list[PopulationGraph] = []
wl_set = set()

def add_new_random_graph(
    graph_zoo: list[PopulationGraph],
    wl_set: set,
    n_nodes: int,
    n_edges: int,
    name: str,
    seed=None,
):

    new_graph, new_wl = None, None
    while new_wl is None or new_wl in wl_set:
        new_graph = PopulationGraph.random_connected_graph(
            n_nodes, n_edges, name=name, seed=seed
        )
        new_wl = new_graph.wl_hash
    graph_zoo.append(new_graph)
    wl_set.add(new_wl)
    return wl_set

for i in range(N_INITIAL_GRAPH_POPULATION):
    # add_new_random_graph(graph_zoo, wl_set, N_NODES, N_EDGES, name=f"random-{i}", seed=int(rng.integers(0, 2**32)))
    new_graph, new_wl = None, None
    while new_wl is None or new_wl in wl_set:
        new_graph = PopulationGraph.random_connected_graph(
            N_NODES, N_EDGES, name=f"random-{i}", seed=None
        )
        new_wl = new_graph.wl_hash
    random_graph_zoo.append(new_graph)
    wl_set.add(new_wl)

In [ ]:
def _model_feature_names(model):
    """Exact feature columns a saved model expects, in order.

    Works for both a bare estimator (XGBoost) and a Pipeline (StandardScaler +
    LinearRegression): sklearn records the training column names on the first
    step. Pulling them from the model itself keeps the GA in lockstep with
    whatever ml_predictors_residual*.ipynb trained, so it never silently feeds
    the wrong / reordered features.
    """
    names = getattr(model, "feature_names_in_", None)
    if names is None and hasattr(model, "named_steps"):
        names = getattr(model[0], "feature_names_in_", None)
    if names is None:
        raise ValueError("Model does not expose feature_names_in_; retrain/re-save it.")
    return list(names)


def _design_matrix(model, props_df):
    """Select the model's features from a properties frame, in the trained order.

    fillna(median) mirrors the training notebook, which imputed medians before
    fitting; a mutated graph can occasionally yield NaN (e.g. undefined
    degree_assortativity), and NaN fitness would poison the argsort selection.
    """
    cols = _model_feature_names(model)
    X = props_df.reindex(columns=cols)
    return X.fillna(X.median())


def run_evolutionary_search_multi_model(
    initial_population: list,
    model,
    model_name,
    secondary_models: dict = None,
    generations: int = 50,
    pop_size: int = 10,
    n_children: int = 50,
    objective: str = "maximize",
    rng: np.random.Generator = None,
):
    """
    Runs a (mu + lambda) evolutionary strategy.

    NOTE: models predict RESIDUALS vs the complete graph
    (delta_prob_fixation, log_ratio_mean_steps), so fitness is centered near 0
    and may be negative. "maximize" -> strongest amplifier / slowest relative to
    the complete graph; "minimize" -> strongest suppressor / fastest.
    """
    if rng is None:
        rng = np.random.default_rng(42)

    # 1. Initialize State
    current_pop = initial_population.copy()
    wl_set = set([g.wl_hash for g in current_pop])
    prop_cache = {}

    # Initialize history dynamically
    history = {model_name: []}
    if secondary_models:
        for key in secondary_models.keys():
            history[key] = []  # Create an empty list for each secondary model

    print(
        f"Starting Evolution: {generations} generations, optimization: {objective} {model_name}"
    )

    for gen in tqdm(range(generations), desc="Evolving"):

        # --- A. REPRODUCTION ---
        children = []
        max_attempts = 10

        for parent in current_pop:
            for i in range(n_children):
                attempts = 0
                while attempts < max_attempts:
                    seed = rng.integers(0, 2**32)
                    new_name = f'{parent.name.split("_")[0]}_gen_{gen}'

                    child = parent.mutate_graph(seed=seed, name=new_name)

                    if child.wl_hash not in wl_set:
                        children.append(child)
                        wl_set.add(child.wl_hash)
                        break
                    attempts += 1

        # --- B. EVALUATION ---
        candidates = current_pop + children

        for g in candidates:
            if g.wl_hash not in prop_cache:
                prop_cache[g.wl_hash] = g.calculate_graph_properties()

        props_df = pd.DataFrame([prop_cache[g.wl_hash] for g in candidates])

        fitness_scores = model.predict(_design_matrix(model, props_df))

        # --- C. SELECTION & SORTING ---
        if objective == "maximize":
            sorted_indices = np.argsort(fitness_scores)[::-1]
        else:
            sorted_indices = np.argsort(fitness_scores)

        top_indices = sorted_indices[:pop_size]
        current_pop = [candidates[i] for i in top_indices]

        # --- D. SECONDARY TRACKING & LOGGING ---
        if secondary_models:
            survivors_props = props_df.iloc[top_indices]
            for key, val in secondary_models.items():
                history[key].append(
                    np.mean(val.predict(_design_matrix(val, survivors_props)))
                )

        # Log primary fitness metrics
        history[model_name].append(np.mean(fitness_scores[top_indices]))

    return current_pop, history

In [ ]:
# def plot_multi_model_history(history, main_model_name, secondary_model_names, objective):
#     if isinstance(secondary_model_names, str):
#         secondary_model_names = [secondary_model_names]

#     fig, ax1 = plt.subplots(figsize=(12, 7))

#     # --- STYLING RULES ---
#     def get_style(model_name):
#         # Determine Color
#         if "LR" in model_name or "Linear Regression" in model_name:
#             color = '#1f77b4' # Blue
#         elif "XGBOOST" in model_name:
#             color = '#d62728' # Red
#         else:
#             color = '#2ca02c' # Green (Fallback)

#         # Determine Pattern
#         if "Probability" in model_name:
#             linestyle = "--" # Dashed line for Probability
#         else:
#             linestyle = "-"  # Solid line for Time

#         return color, linestyle
#     # ---------------------

#     # 1. Determine the metric for the primary (Left) axis
#     main_is_prob = "Probability" in main_model_name

#     # 2. Setup Left Axis Labels
#     ax1.set_xlabel("Generation", fontweight='bold')
#     if main_is_prob:
#         ax1.set_ylabel("Fixation Probability", fontweight='bold', color='black')
#     else:
#         ax1.set_ylabel("Fixation Time (Steps)", fontweight='bold', color='black')

#     lines = []

#     # Plot Main Model Line (Average Only)
#     c_main, ls_main = get_style(main_model_name)

#     # --- THE FIX: Use 'avg_fitness' instead of main_model_name ---
#     l_main = ax1.plot(history[main_model_name], label=f"Avg: {main_model_name} (Main)",
#                       color=c_main, linestyle=ls_main, linewidth=3)
#     lines += l_main
#     # -------------------------------------------------------------

#     # 3. Setup Right Axis Labels
#     ax2 = ax1.twinx()
#     if not main_is_prob:
#         ax2.set_ylabel("Fixation Probability", fontweight='bold', color='black')
#     else:
#         ax2.set_ylabel("Fixation Time (Steps)", fontweight='bold', color='black')

#     # 4. Plot Secondary Models
#     for sec_name in secondary_model_names:
#         sec_is_prob = "Probability" in sec_name

#         # Route to the correct axis based on metric
#         target_ax = ax1 if (sec_is_prob == main_is_prob) else ax2

#         c_sec, ls_sec = get_style(sec_name)
#         l_sec = target_ax.plot(history[sec_name], label=f"Avg: {sec_name}",
#                                color=c_sec, linestyle=ls_sec, linewidth=2.5)
#         lines += l_sec

#     # 5. Set Limits (Ensuring Y-axes strictly start at 0)
#     if main_is_prob:
#         ax1.set_ylim(0, 1.05)
#         ax2.set_ylim(bottom=0)
#     else:
#         ax1.set_ylim(bottom=0)
#         ax2.set_ylim(0, 1.05)

#     # 6. Sort and Create Unified Legend
#     # Zip lines and labels together so they sort in tandem
#     lines_labels = list(zip(lines, [l.get_label() for l in lines]))

#     def legend_sort_key(item):
#         label = item[1]

#         # Priority 1: Model (LR=0, XGBOOST=1, Other=2)
#         if "LR" in label or "Linear Regression" in label:
#             model_sort = 0
#         elif "XGBOOST" in label:
#             model_sort = 1
#         else:
#             model_sort = 2

#         # Priority 2: Unit (Time=0, Probability=1)
#         if "Probability" in label:
#             unit_sort = 1
#         else:
#             unit_sort = 0

#         return (model_sort, unit_sort)

#     # Sort using our custom logic
#     lines_labels.sort(key=legend_sort_key)

#     # Unpack back into lists
#     sorted_lines, sorted_labels = zip(*lines_labels)

#     # Render with the exact same styling as before
#     ax1.legend(sorted_lines, sorted_labels, loc='upper center', bbox_to_anchor=(0.5, -0.15),
#                fancybox=True, shadow=True, ncol=2)

#     # Final Formatting
#     ax1.grid(True, linestyle=':', alpha=0.7)
#     plt.title(f"Evolution of Topologies\nOptimizing: {objective.title()} {main_model_name}", fontsize=14, pad=15)
#     fig.tight_layout()
#     plt.show()

In [ ]:
import matplotlib.pyplot as plt


def plot_multi_model_history(
    history,
    main_model_name,
    secondary_model_names,
    objective,
    rho_complete,
    t_complete,
):
    """Plot avg-survivor fitness per generation, in PHYSICAL units.

    history stores RESIDUAL predictions (the models' native output). We invert
    the residual definitions from add_analytic_reference_columns so the axes read
    in real units:
        Probability model: rho_graph = rho_complete + delta_prob_fixation
        Time model:        T_graph   = t_complete * exp(log_ratio_mean_steps)
    Averaging is done in residual space, so the plotted time is the GEOMETRIC
    mean of survivor fixation times (natural for a log-ratio target); the
    probability curve is an exact arithmetic mean (the delta transform is linear).
    """
    if isinstance(secondary_model_names, str):
        secondary_model_names = [secondary_model_names]

    def to_physical(model_name, residual_series):
        vals = np.asarray(residual_series, dtype=float)
        if "Probability" in model_name:
            return rho_complete + vals  # rho_graph = rho_complete + delta
        return t_complete * np.exp(vals)  # T_graph = t_complete * exp(log-ratio)

    fig, ax1 = plt.subplots(figsize=(12, 7))

    # --- STYLING RULES ---
    def get_style(model_name):
        if "LR" in model_name or "Linear Regression" in model_name:
            color = "#1f77b4"  # Blue
        elif "XGBOOST" in model_name:
            color = "#d62728"  # Red
        else:
            color = "#2ca02c"  # Green (Fallback)
        linestyle = "--" if "Probability" in model_name else "-"
        return color, linestyle

    # 1. Axes are in PHYSICAL units. Left = fixation time, right = fixation prob.
    ax1.set_xlabel("Generation", fontweight="bold")
    ax1.set_ylabel("Mean Fixation Time (steps)", fontweight="bold", color="black")
    ax2 = ax1.twinx()
    ax2.set_ylabel("Fixation Probability", fontweight="bold", color="black")

    lines = []

    def plot_line(model_name, is_main):
        is_prob = "Probability" in model_name
        target_ax = ax2 if is_prob else ax1
        c, ls = get_style(model_name)
        lw = 3.5 if is_main else 2.0  # Thicker line for the optimized model
        label_suffix = " (Main)" if is_main else ""
        return target_ax.plot(
            to_physical(model_name, history[model_name]),
            label=f"Avg: {model_name}{label_suffix}",
            color=c,
            linestyle=ls,
            linewidth=lw,
        )

    # 2. Plot Models
    lines += plot_line(main_model_name, is_main=True)
    for sec_name in secondary_model_names:
        lines += plot_line(sec_name, is_main=False)

    # 3. Complete-graph baselines (the residual origin, drawn in real units).
    ax1.axhline(t_complete, color="#7f7f7f", linestyle=":", linewidth=1.2, alpha=0.8)
    ax1.text(
        0.01,
        t_complete,
        f" complete T={t_complete:.0f}",
        transform=ax1.get_yaxis_transform(),
        va="bottom",
        ha="left",
        fontsize=8,
        color="#7f7f7f",
    )
    ax2.axhline(
        rho_complete, color="#7f7f7f", linestyle=(0, (1, 3)), linewidth=1.2, alpha=0.8
    )
    ax2.text(
        0.99,
        rho_complete,
        f"complete rho={rho_complete:.3f} ",
        transform=ax2.get_yaxis_transform(),
        va="bottom",
        ha="right",
        fontsize=8,
        color="#7f7f7f",
    )

    # 4. Sort and Create Unified Legend
    lines_labels = list(zip(lines, [l.get_label() for l in lines]))

    def legend_sort_key(item):
        label = item[1]
        if "LR" in label or "Linear Regression" in label:
            model_sort = 0
        elif "XGBOOST" in label:
            model_sort = 1
        else:
            model_sort = 2
        unit_sort = 1 if "Probability" in label else 0
        return (model_sort, unit_sort)

    lines_labels.sort(key=legend_sort_key)
    sorted_lines, sorted_labels = zip(*lines_labels)
    ax1.legend(
        sorted_lines,
        sorted_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.15),
        fancybox=True,
        shadow=True,
        ncol=2,
    )

    # 5. Final Formatting
    ax1.grid(True, linestyle=":", alpha=0.7)
    plt.title(
        f"Evolution of Topologies\nOptimizing: {objective.title()} {main_model_name}",
        fontsize=14,
        pad=15,
    )
    fig.tight_layout()
    plt.show()

In [ ]:
# Residual predictors trained on a single r=1.1 slice (ml_predictors_residual_single_r.ipynb).
# Targets are residuals vs the complete graph:
#   log_ratio_mean_steps = log(T_graph / T_complete)      -> "Fixation Time" axis
#   delta_prob_fixation  = rho_graph - rho_complete        -> "Fixation Probability" axis
# The display names keep the "LR"/"XGBOOST"/"Time"/"Probability" tokens the plotting
# helper routes and colors on.
R_TAG = "r1.1"
R_VALUE = 1.1  # selection coefficient the residual models were trained at

TIME_LR_MODEL = "LR Fixation Time (log-ratio)"
TIME_XGBOOST_MODEL = "XGBOOST Fixation Time (log-ratio)"
PROB_LR_MODEL = "LR Fixation Probability (delta)"
PROB_XGBOOST_MODEL = "XGBOOST Fixation Probability (delta)"

OBJECTIVES = [
    "maximize", 
    "minimize"
    ]


models = {
    TIME_LR_MODEL: joblib.load(
        ML_MODELS_DIR / f"log_ratio_mean_steps_{R_TAG}_linear_regression_pipeline.joblib"
    ),
    TIME_XGBOOST_MODEL: joblib.load(
        ML_MODELS_DIR / f"log_ratio_mean_steps_{R_TAG}_xgboost_model.joblib"
    ),
    PROB_LR_MODEL: joblib.load(
        ML_MODELS_DIR / f"delta_prob_fixation_{R_TAG}_linear_regression_pipeline.joblib"
    ),
    PROB_XGBOOST_MODEL: joblib.load(
        ML_MODELS_DIR / f"delta_prob_fixation_{R_TAG}_xgboost_model.joblib"
    ),
}

# Complete-graph baselines at (N=31, r=1.1): the "0" of every residual. The plot
# adds these back to show ML predictions on the physical rho / fixation-time axes.
RHO_COMPLETE = float(analytic_moran_fc_fixation_prob(N_NODES, R_VALUE))
T_COMPLETE = float(analytic_moran_fc_fixation_time(N_NODES, R_VALUE))
print(f"complete-graph baselines @ N={N_NODES}, r={R_VALUE}: "
      f"rho={RHO_COMPLETE:.4f}, T={T_COMPLETE:.1f} steps")

In [ ]:
# Run the Evolutionary Algorithm!


os.makedirs(EXTREME_GRAPH_ZOO_DIR, exist_ok=True)

final_pops = []
for objective in OBJECTIVES:
    for model_name in models.keys():
        secondary_models = {k: v for k, v in models.items() if k != model_name}
        # 1. Configuration
        params = {
            "initial_population": random_graph_zoo,  # Start with your random zoo
            "model": models[model_name],  # Your trained Linear Regression
            "model_name": model_name,
            "secondary_models": secondary_models,
            "generations": GENERATIONS,  # How long to run
            "pop_size": N_INITIAL_GRAPH_POPULATION,  # Keep top 10 elite graphs
            "n_children": NUMBER_OF_CHILDREN,  # Generate 30 new mutatesd graphs per graph in the population
            "objective": objective,
            "rng": rng,
        }
        # 2. Run
        final_pop_1, history = run_evolutionary_search_multi_model(**params)

        plot_multi_model_history(
            history,
            main_model_name=model_name,
            secondary_model_names=list(secondary_models.keys()),
            objective=objective,
            rho_complete=RHO_COMPLETE,
            t_complete=T_COMPLETE,
        )
        winner_graph_zoo_file = Path("graph_zoos") / (
            f'extreme_{objective}_{model_name.replace(" ", "_")}.joblib'
        )
        winner_graph_zoo_file.parent.mkdir(parents=True, exist_ok=True)
        category = f"{objective} {model_name}"
        print(category)
        for graph in final_pop_1:
            graph.category = category

        final_pops.append(final_pop_1)
        # joblib.dump(final_pop_1, winner_graph_zoo_file)

from moran_process.core.graph_zoo import GraphZoo

extreme_zoo = GraphZoo(name="Extreme Graphs (GA-optimized)")
for pop in final_pops:
    for graph in pop:
        extreme_zoo.add(graph)


In [ ]:

if OVERWRITE_EXTREME_GRAPHS:
    extreme_zoo.save(str(EXTREME_GRAPH_ZOO_DIR / "extreme_graphs.pkl"))
    print("Extreme graphs saved.")
else:
    print("OVERWRITE_EXTREME_GRAPHS is False. Skipping save.")

In [ ]:
from moran_process.core.graph_zoo import GraphZoo

pkl_path = EXTREME_GRAPH_ZOO_DIR / "extreme_graphs.pkl"

if not pkl_path.exists():
    raise FileNotFoundError(
        f"extreme_graphs.pkl not found at {pkl_path}\n\n"
        "To generate it, run cell 10 with OVERWRITE_EXTREME_GRAPHS = True"
    )

extreme_zoo = GraphZoo.load(str(pkl_path))

# Index graphs by category
categories = {}
for graph in extreme_zoo:
    cat = getattr(graph, "category", "uncategorized")
    if cat not in categories:
        categories[cat] = []
    categories[cat].append(graph)

print("Available categories:")
for i, (cat, graphs) in enumerate(categories.items()):
    print(f"  [{i}] {cat} ({len(graphs)} graphs)")

# View Extreme Graphs

In [ ]:
# Draw graphs from selected categories
# Modify the list below to select which categories to view
CATEGORIES_TO_VIEW = list(categories.keys())  # View all, or pick specific ones:
# CATEGORIES_TO_VIEW = ["maximize LR Fixation Time (log-ratio)", "minimize LR Fixation Time (log-ratio)"]

for cat in CATEGORIES_TO_VIEW:
    if cat in categories:
        print(f"\n=== {cat} ===")
        for graph in categories[cat]:
            graph.draw(title=graph.name)